# Normalize Article Text

This notebook prepares Khmer article text for search. It applies Unicode and whitespace normalization only. It does not translate, correct legal wording, tokenize, segment, or generate embeddings.

In [5]:
import json
import re
import unicodedata
from pathlib import Path

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'react-client').is_dir() and (path / 'data-pipeline').is_dir()
)
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
INPUT_PATH = DATA_DIR / 'processed' / 'articles.jsonl'
OUTPUT_PATH = DATA_DIR / 'processed' / 'articles_normalized.jsonl'
ZERO_WIDTH = '\u200b\u200c\u200d\ufeff'
print('Input:', INPUT_PATH)
print('Output:', OUTPUT_PATH)

Input: /Users/nak/CADT/projects/legal-rag/data-pipeline/data/processed/articles.jsonl
Output: /Users/nak/CADT/projects/legal-rag/data-pipeline/data/processed/articles_normalized.jsonl


In [6]:
def normalize_khmer(value):
    text = unicodedata.normalize('NFC', str(value or ''))
    text = text.translate(str.maketrans('', '', ZERO_WIDTH))
    text = text.replace('\u00a0', ' ')
    lines = []
    for line in text.splitlines():
        line = re.sub(r'[ \t]+', ' ', line).strip()
        if line:
            lines.append(line)
    return re.sub(r'\n{2,}', '\n', '\n'.join(lines))


In [7]:
records = []
for line in INPUT_PATH.read_text(encoding='utf-8').splitlines():
    record = json.loads(line)
    normalized = dict(record)
    normalized['search_text'] = normalize_khmer(record['text'])
    normalized['normalization_version'] = 'unicode-nfc-v1'
    records.append(normalized)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open('w', encoding='utf-8') as output:
    for record in records:
        output.write(json.dumps(record, ensure_ascii=False) + '\n')

print(f'Normalized articles: {len(records):,}')
print(f'Output: {OUTPUT_PATH}')

Normalized articles: 546
Output: /Users/nak/CADT/projects/legal-rag/data-pipeline/data/processed/articles_normalized.jsonl


In [8]:
assert records
assert len(records) == len({record['id'] for record in records})
assert all(record['search_text'].strip() for record in records)
assert not any(any(character in record['search_text'] for character in ZERO_WIDTH) for record in records)
print('Validation passed')
print(json.dumps(records[0], ensure_ascii=False, indent=2))

Validation passed
{
  "id": "1004-1",
  "article_id": "1004-1",
  "document_id": "1004",
  "article_number": 1,
  "text": "ច្បាប់នេះមានគោលដៅការពារតក្កកម្មដែលទទួលបានប្រកាសនីយបត្រតក្កកម្ម ម៉ូដែលដែលមានអត្ថប្រយោជន៍ដែលទទួលបានវិញ្ញាបនបត្រ និងការការពារគំនូរឧស្សាហកម្ម ដែលបានចុះបញ្ជីនៅក្នុងព្រះរាជាណាចក្រកម្ពុជា ស្របតាមច្បាប់នេះ និងសន្ធិសញ្ញាសហប្រតិបត្តិការប្រកាសនីយបត្រតក្កកម្ម",
  "document_title": "ច្បាប់ស្តីពីប្រកាសនីយបត្រតក្កកម្មវិញ្ញាបនបត្រម៉ូដែល មានអត្ថប្រយោជន៍ និងគំនូរឧស្សាហកម្ម",
  "document_number": "នស/រកម/០១០៣/០០៥",
  "document_type": "Law",
  "source_url": "https://library.ncdd.gov.kh/detail/1004",
  "pdf_url": "https://lib.ncdd.gov.kh/storage/app/public/library_backend/CAT_2105_1/2003-Law-Certificates of Useful Modeling and Industrial Painting.pdf",
  "chapter_number": 1,
  "chapter_title": "បទប្បញ្ញត្តិទូទៅ",
  "search_text": "ច្បាប់នេះមានគោលដៅការពារតក្កកម្មដែលទទួលបានប្រកាសនីយបត្រតក្កកម្ម ម៉ូដែលដែលមានអត្ថប្រយោជន៍ដែលទទួលបានវិញ្ញាបនបត្រ និងការការពារគំនូរឧស្សាហកម្ម ដែលបានចុះបញ្ជីនៅក្ន